[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00


In [4]:
import torch
import math

In [8]:

def flash_attention(Q, K, V, block_size=32): #blocks alon sequence, with ALL DIMS and ALL BATCHES
  B, S, D = K.shape
  Qblocks = [Q[:, i:i+block_size, :] for i in range(0, S, block_size)] #(B, block_size, D) each (last block can have LESS than `block_size` tokens!)
  q_len = len(Qblocks)
  Kblocks = [K[:, i:i+block_size, :] for i in range(0, S, block_size)] #(B, block_size, D) each
  k_len = len(Kblocks)
  Vblocks = [V[:, i:i+block_size, :] for i in range(0, S, block_size)] #(B, block_size, D) each

  Oblocks = []

  for i in range(q_len): #for every Q block

    ### LOAD Qi, Oi, mi, li from HBM to SRAM ###
    Qi = Qblocks[i] #(B, block_size, D)
    Qi_len = Qi.shape[1] #in a full block it is blocks_size but last block can be less if S % block_size != 0

    Oi = torch.zeros((B, Qi_len, D), device=Q.device, dtype=Q.dtype) #(B, block_size, 1)
    mi = torch.full((B, Qi_len, 1), float("-inf"), device=Q.device, dtype=Q.dtype) #(B, block_size, 1)
    li = torch.zeros((B, Qi_len, 1), device=Q.device, dtype=Q.dtype) #(B, block_size, 1)

    for j in range(k_len): #for every K-V block

      ### LOAD Kj, Vj from HBM to SRAM and compute everything on SRAM ###
      Kj, Vj = Kblocks[j], Vblocks[j] #(B, block_size, D)

      ### compute LOCAL stats
      Sij = Qi @ Kj.transpose(-2, -1) / math.sqrt(D) #block scores (B, block_size, block_size)
      mij = torch.max(Sij, dim = -1, keepdim=True).values #(B, block_size, 1), max for each row
      fij = torch.exp(Sij - mij) #exp for online softmax (num of softmax), (B, block_size, block_size)
      lij = torch.sum(fij, dim = -1, keepdim=True) #(B, block_size, 1), sum for each row (den of softmax)

      mi_new = torch.maximum(mi, mij) #(B, block_size, 1)
      old_scale = torch.exp(mi - mi_new) #(B, block_size, 1) --> if mi was not the max, we shifted of -mi each raw_weight wrongly, so we add it back (+mi) and instead we shift with the new found max (-mi_new)
      new_scale = torch.exp(mij - mi_new) #(B, block_size, 1) --> sane but with the new block


      li_new = old_scale * li + new_scale * lij #(B, block_size, 1) --> update running sum resfhiting old and adding correctly shifted new

      # merge old output + new block --> same as li_new but for the numerator of softmax (exp raw_weight shifted)
      Oi = (
          old_scale * li * Oi #Oi was the classim weighted sum of Vs where the denominator was li and the numerator was the exp raw_weight shifted (fij) sum of V
          #so if we multiply li we are simplifying the denominator --> weighted sum with exp(raw_weights - old_max) --> recover numerator
          #then we adjust the weighted sum with the new found max as before
          +
          new_scale * (fij @ Vj)
      ) / li_new

      mi = mi_new
      li = li_new

    #END LOOP over Kj-Vj for a single Qi

    ### UPDATE O in HBM
    Oblocks.append(Oi) #(B, S, D)

  return torch.cat(Oblocks, dim = 1) #concatenate every block

In [9]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [10]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (11.0ms)
  ✅ [2/4] Non-aligned block size (4.8ms)
  ✅ [3/4] Block size invariant (5.7ms)
  ✅ [4/4] Gradient flow (4.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (25.6ms total)
  Progress saved. Run status() to see your dashboard.

